# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

### Problem

The goal of this project is to rank content pages for editorial review so that editors can prioritize which pages to review first.

### Method choice

I will start with Logistic Regression because the target is a binary observed label: whether a content item is declining or not.

I will then compare it with Random Forest as a stronger non-linear model.

The models will produce a probability score for the declining class. I will use this score to rank the content items and compare the ranking with the Week-4 baseline.

I will prefer the simpler model if the more complex model does not provide a meaningful improvement.

### Why this fits the task

The project is a supervised learning problem because the dataset contains the target `is_declining_label`.

The model should produce a ranking score rather than only a yes/no prediction, because the business goal is to decide which pages should be reviewed first.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a client-holdout split. Complete clients will be assigned to either the training set or the test set, so content from the same client does not appear in both sets.

This is important because `client_id` is used for grouping and splitting, not as a model feature. I will use a fixed random seed so that the split is reproducible.

The same held-out test set will be used to evaluate both the Week-4 baseline and the models using the same ranking metric.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Prepared feature dataset from the FlyRank starter repo
DATA_PATH = "../../data/processed/refresh_feature_vector.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Columns:", df.shape[1])

Rows: 30000
Clients: 32
Columns: 52


In [2]:
# Create a list of unique clients
clients = df["client_id"].drop_duplicates()

# Split clients, not individual rows
train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Create train and test datasets
train_df = df[
    df["client_id"].isin(train_clients)
].copy()

test_df = df[
    df["client_id"].isin(test_clients)
].copy()

# Check the split
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

# Verify that no client appears in both sets
client_overlap = (
    set(train_df["client_id"])
    &
    set(test_df["client_id"])
)

print("Client overlap:", len(client_overlap))

Train rows: 26581
Test rows: 3419
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs baseline

I will train the model using the prepared feature vector. The target is `is_declining_label`.

I will exclude identifiers and label-derived columns from the feature set to avoid leakage. Categorical features will be encoded separately from numerical features.

In [3]:
# Target
TARGET = "is_declining_label"

# Columns that must not be used as model features
EXCLUDE_COLUMNS = [
    "client_id",
    "content_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

# Keep only columns that actually exist
EXCLUDE_COLUMNS = [
    col for col in EXCLUDE_COLUMNS
    if col in train_df.columns
]

# Feature columns
FEATURE_COLUMNS = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLUMNS
]

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET].astype(int)

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET].astype(int)

print("Number of features:", len(FEATURE_COLUMNS))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTarget rate - train:", round(y_train.mean(), 4))
print("Target rate - test:", round(y_test.mean(), 4))

Number of features: 47
Training rows: 26581
Test rows: 3419

Target rate - train: 0.5444
Target rate - test: 0.5238


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number", "bool"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)

Numeric features: 36
Categorical features: 11

Categorical columns:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Logistic Regression model
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Train
logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [6]:
# Predict probability of the declining class
logistic_scores = logistic_model.predict_proba(X_test)[:, 1]

print("Number of test scores:", len(logistic_scores))
print("Minimum score:", round(logistic_scores.min(), 4))
print("Maximum score:", round(logistic_scores.max(), 4))

Number of test scores: 3419
Minimum score: 0.0
Maximum score: 1.0


In [7]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(scores))

    # Highest predicted scores first
    order = np.argsort(scores)[::-1]

    top_k_indices = order[:k]

    return y_true[top_k_indices].mean()


precision_20 = precision_at_k(
    y_test.values,
    logistic_scores,
    20
)

precision_50 = precision_at_k(
    y_test.values,
    logistic_scores,
    50
)

print("Logistic Regression Precision@20:", round(precision_20, 4))
print("Logistic Regression Precision@50:", round(precision_50, 4))

Logistic Regression Precision@20: 1.0
Logistic Regression Precision@50: 1.0


In [8]:
# Inspect the top-ranked predictions
top50 = test_df[
    ["client_id", "content_id", "is_declining_label"]
].copy()

top50["model_score"] = logistic_scores

top50 = top50.sort_values(
    "model_score",
    ascending=False
).head(50)

display(top50)

,client_id,content_id,is_declining_label,model_score
25056,client_a88a7902cb,content_6605878eeb0d,1,1.000000
12707,client_bbb965ab0c,content_8e5645f83c08,1,1.000000
19198,client_bbb965ab0c,content_4a519bf8cdbd,1,1.000000
22550,client_bbb965ab0c,content_4351cb73a022,1,1.000000
14621,client_a88a7902cb,content_0390a273c940,1,1.000000
8438,client_bbb965ab0c,content_b5b60616573b,1,1.000000
23606,client_a88a7902cb,content_0f61f4c94440,1,1.000000
1502,client_bbb965ab0c,content_6eeeb0e9f975,1,0.999999
9344,client_a88a7902cb,content_41dd7803de3b,1,0.999999
8692,client_9400f1b21c,content_ce861b52509e,1,0.999998


In [9]:
print("Top 20 actual positives:")
print(top50.head(20)["is_declining_label"].value_counts())

print("\nTop 50 actual positives:")
print(top50.head(50)["is_declining_label"].value_counts())

print("\nUnique top scores:")
print(
    pd.Series(logistic_scores)
    .value_counts()
    .head(10)
)

Top 20 actual positives:
is_declining_label
1    20
Name: count, dtype: int64

Top 50 actual positives:
is_declining_label
1    50
Name: count, dtype: int64

Unique top scores:
0.669597    1
0.630426    1
0.612505    1
0.433459    1
0.470265    1
0.858691    1
0.627388    1
0.558809    1
0.797168    1
0.705552    1
Name: count, dtype: int64


In [10]:
print("Features containing trend:")
print([
    col for col in FEATURE_COLUMNS
    if "trend" in col.lower()
])

print("\nFeatures containing label:")
print([
    col for col in FEATURE_COLUMNS
    if "label" in col.lower()
])

Features containing trend:
[]

Features containing label:
[]


In [11]:
# ==========================================
# LOGISTIC REGRESSION FEATURE INSPECTION
# ==========================================

# Get transformed feature names
fitted_preprocessor = logistic_model.named_steps["preprocessor"]

feature_names = fitted_preprocessor.get_feature_names_out()

# Get logistic coefficients
logistic_estimator = logistic_model.named_steps["model"]

coefficients = logistic_estimator.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "abs_coefficient",
    ascending=False
).reset_index(drop=True)

print("Top 20 features by absolute coefficient:")

display(
    feature_importance.head(20)
)

Top 20 features by absolute coefficient:


,feature,coefficient,abs_coefficient
0,numeric__impressions_last_30d,-36.912814,36.912814
1,numeric__impressions_prev_30d,30.925008,30.925008
2,numeric__log_impressions_90d,1.656504,1.656504
3,numeric__clicks_last_30d,-1.133729,1.133729
4,numeric__clicks_prev_30d,1.053530,1.053530
5,categorical__position_tier_top_3,-0.768003,0.768003
6,categorical__impression_tier_good,-0.622591,0.622591
7,numeric__pageviews_90d,0.606081,0.606081
8,categorical__model_used_gemini-2.5-flash,-0.576986,0.576986
9,categorical__content_type_feedly article,-0.572911,0.572911


In [12]:
print("Top 10 positive features:")
display(
    feature_importance
    .sort_values("coefficient", ascending=False)
    .head(10)
)

print("\nTop 10 negative features:")
display(
    feature_importance
    .sort_values("coefficient", ascending=True)
    .head(10)
)

Top 10 positive features:


,feature,coefficient,abs_coefficient
1,numeric__impressions_prev_30d,30.925008,30.925008
2,numeric__log_impressions_90d,1.656504,1.656504
4,numeric__clicks_prev_30d,1.053530,1.053530
7,numeric__pageviews_90d,0.606081,0.606081
12,numeric__clicks_90d,0.543443,0.543443
16,numeric__impressions_90d,0.435642,0.435642
17,categorical__content_type_keyword article,0.427240,0.427240
19,categorical__impression_tier_low,0.420476,0.420476
20,numeric__sessions_90d,0.420202,0.420202
25,categorical__model_used_gpt-4o-mini,0.328799,0.328799



Top 10 negative features:


,feature,coefficient,abs_coefficient
0,numeric__impressions_last_30d,-36.912814,36.912814
3,numeric__clicks_last_30d,-1.133729,1.133729
5,categorical__position_tier_top_3,-0.768003,0.768003
6,categorical__impression_tier_good,-0.622591,0.622591
8,categorical__model_used_gemini-2.5-flash,-0.576986,0.576986
9,categorical__content_type_feedly article,-0.572911,0.572911
10,numeric__content_age_days,-0.556143,0.556143
11,numeric__users_90d,-0.555213,0.555213
13,numeric__log_clicks_90d,-0.540249,0.540249
14,categorical__word_count_tier_<1000,-0.505251,0.505251


### Initial model diagnostic

The initial Logistic Regression achieved Precision@20 and Precision@50 of 1.00. However, feature inspection showed extremely large coefficients for `impressions_last_30d` and `impressions_prev_30d`.

These variables are directly related to how the declining label is constructed. Therefore, the perfect ranking result is not treated as a fair final model result.

I will retrain the model after removing the last-30-day variables that directly encode the label construction.

In [13]:
# Remove variables that directly encode the label construction
LEAKAGE_FEATURES = [
    "trend_pct",
    "trend_direction",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

CLEAN_FEATURE_COLUMNS = [
    col for col in FEATURE_COLUMNS
    if col not in LEAKAGE_FEATURES
]

X_train_clean = train_df[CLEAN_FEATURE_COLUMNS].copy()
X_test_clean = test_df[CLEAN_FEATURE_COLUMNS].copy()

y_train_clean = train_df[TARGET].astype(int)
y_test_clean = test_df[TARGET].astype(int)

print("Original features:", len(FEATURE_COLUMNS))
print("Clean features:", len(CLEAN_FEATURE_COLUMNS))

print("\nRemoved features:")
print([
    col for col in FEATURE_COLUMNS
    if col in LEAKAGE_FEATURES
])

Original features: 47
Clean features: 44

Removed features:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']


In [14]:
# ==========================================
# CLEAN LOGISTIC REGRESSION
# ==========================================

# Identify numeric and categorical features
numeric_features_clean = X_train_clean.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features_clean = X_train_clean.select_dtypes(
    exclude=["number", "bool"]
).columns.tolist()

print("Clean numeric features:", len(numeric_features_clean))
print("Clean categorical features:", len(categorical_features_clean))


# Numerical preprocessing
numeric_pipeline_clean = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical preprocessing
categorical_pipeline_clean = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# Combine preprocessing
preprocessor_clean = ColumnTransformer([
    ("numeric", numeric_pipeline_clean, numeric_features_clean),
    ("categorical", categorical_pipeline_clean, categorical_features_clean)
])


# Clean Logistic Regression
logistic_model_clean = Pipeline([
    ("preprocessor", preprocessor_clean),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])


# Train
logistic_model_clean.fit(
    X_train_clean,
    y_train_clean
)

print("Clean Logistic Regression training completed.")

Clean numeric features: 33
Clean categorical features: 11
Clean Logistic Regression training completed.


In [15]:
# Predict probability of the declining class
clean_scores = logistic_model_clean.predict_proba(
    X_test_clean
)[:, 1]

print("Number of clean model scores:", len(clean_scores))
print("Minimum score:", round(clean_scores.min(), 4))
print("Maximum score:", round(clean_scores.max(), 4))

Number of clean model scores: 3419
Minimum score: 0.0046
Maximum score: 0.9483


In [16]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(scores))

    order = np.argsort(scores)[::-1]
    top_k_indices = order[:k]

    return y_true[top_k_indices].mean()


clean_precision_20 = precision_at_k(
    y_test_clean.values,
    clean_scores,
    20
)

clean_precision_50 = precision_at_k(
    y_test_clean.values,
    clean_scores,
    50
)

print(
    "Clean Logistic Regression Precision@20:",
    round(clean_precision_20, 4)
)

print(
    "Clean Logistic Regression Precision@50:",
    round(clean_precision_50, 4)
)

Clean Logistic Regression Precision@20: 0.65
Clean Logistic Regression Precision@50: 0.7


In [17]:
# Check the actual position/impression columns available
print([
    col for col in test_df.columns
    if "position" in col.lower() or "impression" in col.lower()
])

['impressions_90d', 'days_with_impressions', 'impressions_last_30d', 'impressions_prev_30d', 'avg_position', 'impression_tier', 'position_tier', 'log_impressions_90d']


In [18]:
print("Columns containing 30d:")
print([
    col for col in test_df.columns
    if "30d" in col.lower()
])

Columns containing 30d:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


In [19]:
# ==========================================
# LOAD THE EXACT WEEK-4 BASELINE
# ==========================================

import os
import pandas as pd

possible_paths = [
    "../outputs/baseline_action_score.csv",
    "../../work/outputs/baseline_action_score.csv",
    "work/outputs/baseline_action_score.csv"
]

baseline_path = None

for path in possible_paths:
    if os.path.exists(path):
        baseline_path = path
        break

print("Baseline path:", baseline_path)

if baseline_path is None:
    print("\nBaseline CSV not found in the checked locations.")
else:
    baseline_df = pd.read_csv(baseline_path)

    print("Baseline rows:", len(baseline_df))
    print("\nBaseline columns:")
    print(baseline_df.columns.tolist())

    display(baseline_df.head())

Baseline path: work/outputs/baseline_action_score.csv
Baseline rows: 176738

Baseline columns:
['client_hash_id', 'content_hash_id', 'impressions_30d', 'avg_position_30d', 'score', 'reason_code', 'action']


C:\Users\rajku\AppData\Local\Temp\ipykernel_3736\2327062318.py:26: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  baseline_df = pd.read_csv(baseline_path)


,client_hash_id,content_hash_id,impressions_30d,avg_position_30d,score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,15.008339,1,visible_position_risk,review_for_optimization
1,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,32.766674,1,visible_position_risk,review_for_optimization
2,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,22.558608,1,visible_position_risk,review_for_optimization
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,23.335465,1,visible_position_risk,review_for_optimization
4,client_23a62021009f63c4,content_66288edeb93b7c4f,137878.0,18.615742,1,visible_position_risk,review_for_optimization


In [20]:
# Check all columns related to position and impressions
baseline_candidates = [
    col for col in test_df.columns
    if any(word in col.lower() for word in [
        "impression",
        "position",
        "gsc"
    ])
]

print("Possible baseline-related columns:")
for col in baseline_candidates:
    print(col)

Possible baseline-related columns:
impressions_90d
days_with_impressions
impressions_last_30d
impressions_prev_30d
avg_position
impression_tier
position_tier
log_impressions_90d


### Baseline comparison note

The Week-4 baseline used `impressions_30d` and `avg_position_30d`. The current 30,000-row starter dataset does not contain `avg_position_30d`, so the exact Week-4 baseline cannot be reproduced on this client-holdout test set.

I therefore do not substitute `avg_position` for `avg_position_30d`, because that would change the baseline definition. The clean Logistic Regression results are reported separately, and the baseline mismatch is treated as a limitation rather than an invented comparison.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
# ==========================================
# 4. ERRORS AND INTERPRETATION
# ==========================================

# Create test results table
error_df = test_df[
    ["client_id", "content_id", TARGET]
].copy()

error_df["model_score"] = clean_scores

# Rank by model score
error_df = error_df.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

# Use 0.5 as classification threshold for error analysis
error_df["predicted_label"] = (
    error_df["model_score"] >= 0.5
).astype(int)

# Error flag
error_df["error"] = (
    error_df["predicted_label"] != error_df[TARGET]
)

print("Total test rows:", len(error_df))
print("Total errors:", error_df["error"].sum())
print(
    "Error rate:",
    round(error_df["error"].mean(), 4)
)

Total test rows: 3419
Total errors: 1334
Error rate: 0.3902


In [22]:
# ==========================================
# FALSE POSITIVES AND FALSE NEGATIVES
# ==========================================

false_positives = error_df[
    (error_df["predicted_label"] == 1) &
    (error_df[TARGET] == 0)
].copy()

false_negatives = error_df[
    (error_df["predicted_label"] == 0) &
    (error_df[TARGET] == 1)
].copy()

true_positives = error_df[
    (error_df["predicted_label"] == 1) &
    (error_df[TARGET] == 1)
].copy()

true_negatives = error_df[
    (error_df["predicted_label"] == 0) &
    (error_df[TARGET] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))
print("True positives:", len(true_positives))
print("True negatives:", len(true_negatives))

False positives: 964
False negatives: 370
True positives: 1421
True negatives: 664


In [23]:
# ==========================================
# 3 CONCRETE WRONG CASES
# ==========================================

print("FALSE POSITIVES — top 3")
display(
    false_positives
    .sort_values("model_score", ascending=False)
    .head(3)
)

print("FALSE NEGATIVES — top 3")
display(
    false_negatives
    .sort_values("model_score", ascending=False)
    .head(3)
)

FALSE POSITIVES — top 3


,client_id,content_id,is_declining_label,model_score,predicted_label,error
4,client_8527a891e2,content_5ce1a9d3e4d7,0,0.925631,1,True
5,client_8527a891e2,content_41baf0722ad9,0,0.920678,1,True
8,client_8527a891e2,content_35d63627bf3e,0,0.914900,1,True


FALSE NEGATIVES — top 3


,client_id,content_id,is_declining_label,model_score,predicted_label,error
2386,client_a88a7902cb,content_97232fb7d9a8,1,0.499752,0,True
2387,client_a88a7902cb,content_f10e5fbd5d00,1,0.499520,0,True
2395,client_bbb965ab0c,content_96a6d08e86be,1,0.498216,0,True


In [24]:
# ==========================================
# INSPECT FEATURES OF WRONG CASES
# ==========================================

wrong_cases = pd.concat([
    false_positives.sort_values("model_score", ascending=False).head(3),
    false_negatives.sort_values("model_score", ascending=False).head(3)
])

# Keep only the IDs needed to find original rows
wrong_ids = wrong_cases[
    ["client_id", "content_id", TARGET, "model_score", "predicted_label"]
].copy()

# Bring back selected original features
inspection_columns = [
    "client_id",
    "content_id",
    TARGET,
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "avg_position",
    "content_type",
    "competition_level",
    "main_intent",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

available_columns = [
    col for col in inspection_columns
    if col in test_df.columns
]

wrong_feature_cases = test_df[available_columns].merge(
    wrong_ids,
    on=["client_id", "content_id", TARGET],
    how="inner"
)

display(wrong_feature_cases)

,client_id,content_id,is_declining_label,content_age_days,impressions_90d,clicks_90d,sessions_90d,pageviews_90d,users_90d,avg_position,content_type,competition_level,main_intent,freshness_tier,word_count_tier,impression_tier,position_tier,model_score,predicted_label
0,client_8527a891e2,content_5ce1a9d3e4d7,0,223,8076,6,24,26,23,8.1,keyword article,LOW,informational,91-180,1000-2000,good,page_1,0.925631,1
1,client_8527a891e2,content_35d63627bf3e,0,238,1525,0,2,2,2,32.6,keyword article,LOW,commercial,91-180,1000-2000,moderate,page_3_5,0.914900,1
2,client_a88a7902cb,content_97232fb7d9a8,1,180,749,4,63,79,40,23.8,keyword article,LOW,informational,0-30,2000-3500,moderate,page_3_5,0.499752,0
3,client_8527a891e2,content_41baf0722ad9,0,275,3115,0,4,4,4,12.8,keyword article,LOW,informational,91-180,1000-2000,good,striking,0.920678,1
4,client_bbb965ab0c,content_96a6d08e86be,1,133,3324,23,39,39,39,7.9,keyword article,LOW,informational,0-30,2000-3500,good,page_1,0.498216,0
5,client_a88a7902cb,content_f10e5fbd5d00,1,112,318,0,3,3,3,47.9,keyword article,LOW,informational,0-30,2000-3500,moderate,page_3_5,0.499520,0


### Error analysis

The model produced 964 false positives and 370 false negatives on the test set.

Three concrete errors show why individual cases are difficult. One false positive had high impressions and a page-1 position but was still assigned a high declining score. Another false positive had weak impressions, engagement, and position signals that resembled patterns associated with decline. A false negative received a score of 0.4998, almost exactly at the 0.5 classification threshold, showing that the model was uncertain about this case.

These examples suggest that the model's errors are concentrated around cases where several signals point in different directions rather than being cleanly separable.

In [25]:
# ==========================================
# CLEAN MODEL — TOP FEATURES
# ==========================================

clean_preprocessor = logistic_model_clean.named_steps["preprocessor"]
clean_estimator = logistic_model_clean.named_steps["model"]

clean_feature_names = (
    clean_preprocessor.get_feature_names_out()
)

clean_coefficients = clean_estimator.coef_[0]

clean_feature_importance = pd.DataFrame({
    "feature": clean_feature_names,
    "coefficient": clean_coefficients,
    "abs_coefficient": np.abs(clean_coefficients)
}).sort_values(
    "abs_coefficient",
    ascending=False
).reset_index(drop=True)

display(clean_feature_importance.head(15))

,feature,coefficient,abs_coefficient
0,numeric__impressions_90d,-3.216228,3.216228
1,numeric__impressions_prev_30d,3.088884,3.088884
2,numeric__log_impressions_90d,1.644896,1.644896
3,numeric__users_90d,-1.553916,1.553916
4,numeric__sessions_90d,1.395172,1.395172
5,numeric__log_clicks_90d,-0.688685,0.688685
6,categorical__impression_tier_low,0.649383,0.649383
7,categorical__freshness_tier_31-90,-0.597828,0.597828
8,numeric__word_count,0.583652,0.583652
9,categorical__content_type_keyword article,0.577618,0.577618


### Feature interpretation

The three strongest features by absolute Logistic Regression coefficient were `impressions_90d`, `impressions_prev_30d`, and `log_impressions_90d`.

`impressions_90d` had the largest absolute coefficient (-3.216), while `impressions_prev_30d` had a positive coefficient (3.089). `log_impressions_90d` was the third strongest feature with a positive coefficient (1.645).

These coefficients describe how the fitted model associates each feature with the predicted declining class while the other model features are held constant. They should not be interpreted as causal effects.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.